# Southern OC Weekly Event Finder (Wed → Tue)

This Colab notebook pulls **concerts**, **plays/theatre**, and **other notable events** happening in the upcoming **Wednesday → Tuesday** window, then ranks and prints the **top 10** based on:

- **Proximity** to a Southern Orange County “home base”
- A heuristic **“compelling” score** (venue prominence + optional Wikipedia popularity + basic quality rules)

It also tries to **avoid**:
- Events meant primarily for **babies/young kids**
- **Lead‑gen / “ad disguised as an event”** (free trials, open houses, info sessions, etc.)
- Low‑signal **nightclub / DJ nights** and **restaurant promos**

> ⚠️ Scraping note: this notebook uses a mix of **official APIs (recommended)** and **lightweight scraping** from a small number of high‑quality venue pages. Always comply with each site’s Terms/robots and keep request volume low.


In [ ]:
# Dependencies
# - In Google Colab, this notebook will install required packages automatically.
# - In GitHub Actions / local runs, install via requirements.txt (so we skip installs here).

import sys
import subprocess
import os
import re
import math
import json
import time
from dataclasses import dataclass, asdict
from datetime import datetime, date, timedelta, time as dtime
from zoneinfo import ZoneInfo
from typing import Optional, List, Dict, Any, Tuple

def _pip_install(pkgs: List[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    _pip_install([
        "requests",
        "beautifulsoup4",
        "pandas",
        "python-dateutil",
        "geopy",
        "tqdm",
        "tenacity",
        "diskcache",
        "lxml",
    ])

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from geopy.distance import geodesic
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from diskcache import Cache

pd.set_option("display.max_colwidth", 120)

# Small on-disk cache (persists during runtime; helps avoid re-hitting Wikipedia/geocoding repeatedly)
cache = Cache("./.cache_event_finder")


## 1) Configuration

Set your **home base** (used for distance ranking) and optionally add API keys.

- **Ticketmaster Discovery API key (recommended):** sign up for a free key and set it below.
- Optional: **Google Places API key** can be used to fetch venue rating counts (kept optional because it requires billing).


In [ ]:
#@title Configuration

# Southern Orange County "home base" (default: Laguna Niguel area)
HOME_LABEL = "Laguna Niguel, CA"
HOME_LAT = 33.5225   #@param {type:"number"}
HOME_LON = -117.7076 #@param {type:"number"}

# How far out to search (miles)
RADIUS_MILES = 45  #@param {type:"integer"}

# If you run this on Tuesday, it will target Wed→Tue automatically.
# You can override dates manually below.
USE_MANUAL_DATES = False  #@param {type:"boolean"}
MANUAL_START = "2025-12-24"  #@param {type:"string"}  # YYYY-MM-DD
MANUAL_END   = "2025-12-30"  #@param {type:"string"}  # YYYY-MM-DD

# Ticketmaster Discovery API (recommended primary source)
TICKETMASTER_API_KEY = ""  #@param {type:"string"}

# Optional (not required): Google Places (venue ratings & review counts)
GOOGLE_PLACES_API_KEY = ""  #@param {type:"string"}

# Output: how many top events to show in total
TOP_N = 10  #@param {type:"integer"}

# Also pull concerts from a curated list of major venues in LA / OC / San Diego.
INCLUDE_MAJOR_VENUE_CONCERTS = True  #@param {type:\"boolean\"}
MAJOR_VENUE_TOP_N = 10  #@param {type:\"integer\"}

# If you schedule this notebook to run daily, enable the guard so it only produces results on Tuesdays.
ENABLE_TUESDAY_GUARD = False  #@param {type:"boolean"}

TZ = "America/Los_Angeles"

# Env overrides (useful for GitHub Actions / secrets)
HOME_LABEL = os.getenv("HOME_LABEL", HOME_LABEL)
HOME_LAT = float(os.getenv("HOME_LAT", HOME_LAT))
HOME_LON = float(os.getenv("HOME_LON", HOME_LON))
RADIUS_MILES = int(os.getenv("RADIUS_MILES", RADIUS_MILES))
USE_MANUAL_DATES = os.getenv("USE_MANUAL_DATES", str(USE_MANUAL_DATES)).lower() in ("1", "true", "yes", "y")
MANUAL_START = os.getenv("MANUAL_START", MANUAL_START)
MANUAL_END = os.getenv("MANUAL_END", MANUAL_END)
TICKETMASTER_API_KEY = os.getenv("TICKETMASTER_API_KEY", TICKETMASTER_API_KEY)
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY", GOOGLE_PLACES_API_KEY)
TOP_N = int(os.getenv("TOP_N", TOP_N))
INCLUDE_MAJOR_VENUE_CONCERTS = os.getenv(\"INCLUDE_MAJOR_VENUE_CONCERTS\", str(INCLUDE_MAJOR_VENUE_CONCERTS)).lower() in (\"1\", \"true\", \"yes\", \"y\")
MAJOR_VENUE_TOP_N = int(os.getenv(\"MAJOR_VENUE_TOP_N\", MAJOR_VENUE_TOP_N))
ENABLE_TUESDAY_GUARD = os.getenv("ENABLE_TUESDAY_GUARD", str(ENABLE_TUESDAY_GUARD)).lower() in ("1", "true", "yes", "y")
TZ = os.getenv("TZ", TZ)


In [ ]:
# Optional: guard so a daily schedule only produces output on Tuesday
if ENABLE_TUESDAY_GUARD:
    run_date = datetime.now(ZoneInfo(TZ)).date()
    if run_date.weekday() != 1:  # Tuesday
        print(f"Not Tuesday ({run_date:%A}); exiting early.")
        raise SystemExit
    print("Tuesday run: continuing.")
else:
    print("Tuesday guard disabled (normal interactive runs).")


In [ ]:
def compute_wed_to_tue_window(run_date: Optional[date] = None, tz: str = TZ) -> Tuple[datetime, datetime]:
    """Return (start_dt, end_dt) for upcoming Wed→Tue window in local tz.

    - If run on Tuesday: start = Wednesday (next day), end = next Tuesday (6 days after start).
    - Otherwise: start = next Wednesday after run_date, end = following Tuesday.
    """
    if run_date is None:
        run_date = datetime.now(ZoneInfo(tz)).date()

    # weekday: Mon=0 ... Sun=6
    if run_date.weekday() == 1:  # Tuesday
        start = run_date + timedelta(days=1)
    else:
        # next Wednesday (weekday=2)
        days_ahead = (2 - run_date.weekday()) % 7
        if days_ahead == 0:
            days_ahead = 7  # if today is Wednesday, take the *next* Wednesday
        start = run_date + timedelta(days=days_ahead)

    end = start + timedelta(days=6)  # Wednesday -> Tuesday inclusive
    start_dt = datetime.combine(start, dtime(0, 0), ZoneInfo(tz))
    end_dt = datetime.combine(end, dtime(23, 59, 59), ZoneInfo(tz))
    return start_dt, end_dt

def get_target_window() -> Tuple[datetime, datetime]:
    if USE_MANUAL_DATES:
        s = dateparser.parse(MANUAL_START).date()
        e = dateparser.parse(MANUAL_END).date()
        return (
            datetime.combine(s, dtime(0,0), ZoneInfo(TZ)),
            datetime.combine(e, dtime(23,59,59), ZoneInfo(TZ)),
        )
    return compute_wed_to_tue_window()

START_DT, END_DT = get_target_window()
print(f"Target window (local): {START_DT}  →  {END_DT}")


In [ ]:
MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

def _clean_ordinals(s: str) -> str:
    return re.sub(r"(\d)(st|nd|rd|th)\b", r"\1", s, flags=re.I)

def parse_date_span(text: str, default_year: Optional[int] = None) -> Tuple[Optional[date], Optional[date]]:
    """Parse strings like:
      - 'December 6 - 28, 2025'
      - 'January 14 – February 1, 2026'
      - 'Feb. 20-Mar. 8, 2026'
      - 'December 31, 2025'
    Returns (start_date, end_date).
    """
    if not text:
        return None, None
    t = _clean_ordinals(text)
    t = t.replace("\u2013", "-").replace("\u2014", "-")  # en/em dash
    t = t.replace("–", "-").replace("—", "-")
    t = re.sub(r"\s+", " ", t).strip()
    # common connectors
    t = t.replace(" to ", " - ")

    # If no dash, parse single date
    if "-" not in t:
        try:
            d = dateparser.parse(t, fuzzy=True, default=datetime(default_year or datetime.now().year, 1, 1)).date()
            return d, d
        except Exception:
            return None, None

    left, right = [x.strip() for x in t.split("-", 1)]

    # If right has no month, inherit from left
    left_m = re.search(r"\b([A-Za-z]{3,9})\b", left)
    right_m = re.search(r"\b([A-Za-z]{3,9})\b", right)
    if left_m and not right_m:
        # prepend month name
        right = f"{left_m.group(1)} {right}"

    # If right has no year, inherit from left if present
    left_y = re.search(r"\b(20\d{2}|19\d{2})\b", left)
    right_y = re.search(r"\b(20\d{2}|19\d{2})\b", right)
    if left_y and not right_y:
        right = f"{right}, {left_y.group(1)}"

    # Try parse both
    try:
        sd = dateparser.parse(left, fuzzy=True, default=datetime(default_year or datetime.now().year, 1, 1)).date()
    except Exception:
        sd = None
    try:
        ed = dateparser.parse(right, fuzzy=True, default=datetime(default_year or datetime.now().year, 1, 1)).date()
    except Exception:
        ed = None

    return sd, ed

def parse_time_text(t: str) -> Optional[dtime]:
    if not t:
        return None
    try:
        # dateutil can parse time-only if we give a default date
        dt = dateparser.parse(t, fuzzy=True, default=datetime(2000,1,1,0,0))
        return dt.time()
    except Exception:
        return None

def overlaps_window(start: Optional[datetime], end: Optional[datetime], window_start: datetime, window_end: datetime) -> bool:
    if start is None:
        return False
    if end is None:
        end = start
    return (start <= window_end) and (end >= window_start)

@dataclass
class Event:
    title: str
    category: str  # 'Concert', 'Play', 'Event'
    start: Optional[datetime]
    end: Optional[datetime]
    venue: Optional[str] = None
    city: Optional[str] = None
    region: Optional[str] = None
    address: Optional[str] = None
    lat: Optional[float] = None
    lon: Optional[float] = None
    source: Optional[str] = None
    url: Optional[str] = None
    description: Optional[str] = None
    price_min: Optional[float] = None
    price_max: Optional[float] = None
    age_hint: Optional[str] = None  # e.g., '12+'
    tags: Optional[str] = None

def event_to_row(e: Event) -> Dict[str, Any]:
    d = asdict(e)
    return d


In [ ]:
# Major venues to always check (LA / OC / San Diego)
# Note: Ticketmaster naming can vary; "tm_keyword"/"tm_city" helps resolve the correct venueId.
MAJOR_CONCERT_VENUES = [
    # Orange County
    {"name": "The Observatory (Santa Ana)", "region": "OC",
     "aliases": ["The Observatory", "Observatory OC", "The Observatory OC", "Constellation Room"],
     "tm_keyword": "The Observatory", "tm_city": "Santa Ana",
     "lat": 33.7133, "lon": -117.9236, "prominence": 0.82},

    {"name": "Honda Center", "region": "OC",
     "aliases": ["Honda Center, Anaheim"],
     "tm_keyword": "Honda Center", "tm_city": "Anaheim",
     "lat": 33.8070, "lon": -117.8766, "prominence": 0.92},

    # San Diego
    {"name": "The Observatory North Park", "region": "San Diego",
     "aliases": ["Observatory North Park", "The Observatory (North Park)"],
     "tm_keyword": "The Observatory North Park", "tm_city": "San Diego",
     "lat": 32.7480, "lon": -117.1337, "prominence": 0.82},

    # Los Angeles
    {"name": "Kia Forum", "region": "LA",
     "aliases": ["The Forum", "The Kia Forum"],
     "tm_keyword": "Kia Forum", "tm_city": "Inglewood",
     "lat": 33.9583, "lon": -118.3419, "prominence": 0.96},

    {"name": "Los Angeles Memorial Coliseum", "region": "LA",
     "aliases": ["The Coliseum", "LA Memorial Coliseum", "LA Coliseum"],
     "tm_keyword": "Los Angeles Memorial Coliseum", "tm_city": "Los Angeles",
     "lat": 34.0141, "lon": -118.2879, "prominence": 0.96},

    {"name": "Hollywood Palladium", "region": "LA",
     "aliases": ["The Palladium", "Palladium"],
     "tm_keyword": "Hollywood Palladium", "tm_city": "Los Angeles",
     "lat": 34.0983, "lon": -118.3267, "prominence": 0.90},

    {"name": "The Fonda Theatre", "region": "LA",
     "aliases": ["Fonda Theatre", "Henry Fonda Theater", "Henry Fonda Theatre"],
     "tm_keyword": "Fonda Theatre", "tm_city": "Los Angeles",
     "lat": 34.1017, "lon": -118.3258, "prominence": 0.88},

    {"name": "Troubadour", "region": "LA",
     "aliases": ["The Troubadour"],
     "tm_keyword": "Troubadour", "tm_city": "West Hollywood",
     "lat": 34.0906, "lon": -118.3860, "prominence": 0.88},

    {"name": "Hollywood Bowl", "region": "LA",
     "aliases": [],
     "tm_keyword": "Hollywood Bowl", "tm_city": "Hollywood",
     "lat": 34.1122, "lon": -118.3391, "prominence": 1.00},

    {"name": "Greek Theatre", "region": "LA",
     "aliases": ["The Greek Theatre"],
     "tm_keyword": "Greek Theatre", "tm_city": "Los Angeles",
     "lat": 34.1193, "lon": -118.3004, "prominence": 0.96},

    {"name": "The Wiltern", "region": "LA",
     "aliases": ["Wiltern Theatre", "The Wiltern Theatre"],
     "tm_keyword": "The Wiltern", "tm_city": "Los Angeles",
     "lat": 34.0626, "lon": -118.3086, "prominence": 0.92},

    {"name": "Dolby Theatre", "region": "LA",
     "aliases": [],
     "tm_keyword": "Dolby Theatre", "tm_city": "Hollywood",
     "lat": 34.1020, "lon": -118.3406, "prominence": 0.92},

    {"name": "Walt Disney Concert Hall", "region": "LA",
     "aliases": ["Disney Concert Hall"],
     "tm_keyword": "Walt Disney Concert Hall", "tm_city": "Los Angeles",
     "lat": 34.0554, "lon": -118.2498, "prominence": 1.00},

    {"name": "The Bellwether", "region": "LA",
     "aliases": [],
     "tm_keyword": "The Bellwether", "tm_city": "Los Angeles",
     "lat": 34.0516, "lon": -118.2509, "prominence": 0.86},
]

# Known venue coordinates (helps avoid geocoding)
KNOWN_VENUES = {
    # Southern OC / nearby
    "South Coast Repertory": (33.6897, -117.8852),  # Costa Mesa (approx)
    "Laguna Playhouse": (33.5422, -117.7845),       # Laguna Beach (approx)
    "Renée and Henry Segerstrom Concert Hall": (33.6902, -117.8880),  # Segerstrom Center campus
    "Samueli Theater": (33.6902, -117.8880),
    "Segerstrom Stage": (33.6897, -117.8852),
    "Julianne Argyros Stage": (33.6897, -117.8852),
    "Soka Performing Arts Center": (33.5696, -117.7251),  # Aliso Viejo (approx)
    "House of Blues Anaheim": (33.8036, -117.9133),
    "The Parish at House of Blues Anaheim": (33.8036, -117.9133),
    "Irvine Barclay Theatre": (33.6455, -117.8433),
}

# Add major venues into KNOWN_VENUES (including aliases)
for v in MAJOR_CONCERT_VENUES:
    KNOWN_VENUES[v["name"]] = (v["lat"], v["lon"])
    for a in v.get("aliases", []) or []:
        KNOWN_VENUES[a] = (v["lat"], v["lon"])

# Venue prominence (0..1); helps “compelling” scoring
VENUE_PROMINENCE = {
    # Local heavy-hitters
    "Renée and Henry Segerstrom Concert Hall": 1.00,
    "South Coast Repertory": 0.95,
    "Segerstrom Stage": 0.95,
    "Julianne Argyros Stage": 0.90,
    "Irvine Barclay Theatre": 0.85,
    "Laguna Playhouse": 0.80,
    "House of Blues Anaheim": 0.80,
    "Soka Performing Arts Center": 0.75,
}

# Add major venues prominence (including aliases)
for v in MAJOR_CONCERT_VENUES:
    p = float(v.get("prominence", 0.9))
    VENUE_PROMINENCE[v["name"]] = max(VENUE_PROMINENCE.get(v["name"], 0.4), p)
    for a in v.get("aliases", []) or []:
        VENUE_PROMINENCE[a] = max(VENUE_PROMINENCE.get(a, 0.4), p)

# Precomputed matchers for labeling
_MAJOR_MATCHERS = []
for v in MAJOR_CONCERT_VENUES:
    _MAJOR_MATCHERS.append(v["name"].lower())
    _MAJOR_MATCHERS.extend([a.lower() for a in (v.get("aliases") or [])])
    # include keyword form as a matcher too
    if v.get("tm_keyword"):
        _MAJOR_MATCHERS.append(str(v["tm_keyword"]).lower())

def is_major_venue_name(venue: Optional[str]) -> bool:
    if not venue:
        return False
    vl = venue.lower()
    return any(m in vl for m in _MAJOR_MATCHERS)

def major_venue_region(venue: Optional[str]) -> Optional[str]:
    if not venue:
        return None
    vl = venue.lower()
    for v in MAJOR_CONCERT_VENUES:
        names = [v["name"]] + list(v.get("aliases") or [])
        if any(n.lower() in vl for n in names):
            return v.get("region")
    return None

def infer_coords(venue: Optional[str], address: Optional[str] = None) -> Tuple[Optional[float], Optional[float]]:
    if venue:
        for k, (lat, lon) in KNOWN_VENUES.items():
            if k.lower() in venue.lower():
                return lat, lon
        if venue in KNOWN_VENUES:
            return KNOWN_VENUES[venue]
    return None, None


In [ ]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type(Exception),
)
def geocode_address(addr: str) -> Tuple[Optional[float], Optional[float]]:
    """Geocode an address using Nominatim (OpenStreetMap)."""
    if not addr:
        return None, None
    cache_key = f"geocode::{addr}"
    if cache_key in cache:
        return cache[cache_key]

    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": addr, "format": "json", "limit": 1}
    headers = {"User-Agent": "soc-weekly-event-finder/1.0 (personal use)"}
    r = requests.get(url, params=params, headers=headers, timeout=20)
    r.raise_for_status()
    data = r.json()
    if not data:
        cache[cache_key] = (None, None)
        return None, None
    lat, lon = float(data[0]["lat"]), float(data[0]["lon"])
    cache[cache_key] = (lat, lon)
    # be polite
    time.sleep(1.0)
    return lat, lon


In [ ]:
# Heuristic filters
KIDS_KEYWORDS = [
    "baby", "babies", "toddler", "toddlers", "preschool", "pre-school", "storybook", "storytime",
    "sensory", "mommy", "daddy", "parent & me", "parent-and-me", "kids", "kid-friendly",
    "children", "childrens", "little ones", "ages 0", "ages 1", "ages 2", "ages 3", "ages 4",
    "kindergarten", "playdate",
]

PROMO_KEYWORDS = [
    "open house", "free trial", "trial class", "intro class", "information session", "info session",
    "seminar", "webinar", "workshop", "bootcamp", "enroll", "sign up", "signup", "membership",
    "networking", "mixer", "sales pitch", "consultation", "demo", "sample", "grand opening",
    "vendor fair", "market your", "leadership training", "real estate", "insurance",
]

NIGHTLIFE_KEYWORDS = [
    "dj", "dance party", "rave", "after party", "afterparty", "club night", "ladies night",
    "bottle service", "guest list", "happy hour", "karaoke", "open mic", "brunch party",
    "bar crawl", "pub crawl", "kpop night", "taylor swift night", "beyonce night",
]

LOW_SIGNAL_KEYWORDS = [
    "parking", "membership", "mailing list", "meet & greet", "meet and greet", "vip table",
]

def _txt(e: Event) -> str:
    parts = [e.title or "", e.venue or "", e.description or "", e.tags or "", e.age_hint or ""]
    return " ".join([p for p in parts if p]).lower()

def is_kids_event(e: Event) -> bool:
    t = _txt(e)
    if any(k in t for k in KIDS_KEYWORDS):
        # allow if it's clearly teen-appropriate (rare). Keep it simple: exclude.
        return True
    # If we have explicit age hint like '4+' treat as kids.
    if e.age_hint:
        m = re.search(r"(\d{1,2})\s*\+", e.age_hint)
        if m and int(m.group(1)) <= 8:
            return True
    return False

def is_promo_event(e: Event) -> bool:
    t = _txt(e)
    return any(k in t for k in PROMO_KEYWORDS)

def is_nightlife_event(e: Event) -> bool:
    t = _txt(e)
    return any(k in t for k in NIGHTLIFE_KEYWORDS)

def is_low_signal(e: Event) -> bool:
    t = _txt(e)
    return any(k in t for k in LOW_SIGNAL_KEYWORDS)


In [ ]:
def distance_miles(lat: float, lon: float) -> float:
    return geodesic((HOME_LAT, HOME_LON), (lat, lon)).miles

def proximity_score(dist_mi: float) -> float:
    # 0 miles -> 100, 50 miles -> ~0
    return max(0.0, 100.0 - (dist_mi * 2.0))

def venue_prominence(venue: Optional[str]) -> float:
    if not venue:
        return 0.4
    # exact / substring match
    best = 0.4
    for k, v in VENUE_PROMINENCE.items():
        if k.lower() in venue.lower():
            best = max(best, v)
    return best

@retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=1, max=4))
def wikipedia_pageviews_score(query: str, days: int = 30) -> float:
    """Return a 0..1 fame score using Wikipedia pageviews (optional).
    If not found, returns 0.
    """
    q = (query or "").strip()
    if not q:
        return 0.0

    cache_key = f"wiki::{q.lower()}::{days}"
    if cache_key in cache:
        return cache[cache_key]

    # Search
    s_url = "https://en.wikipedia.org/w/api.php"
    s_params = {
        "action": "query",
        "list": "search",
        "srsearch": q,
        "format": "json",
        "srlimit": 1,
    }
    r = requests.get(s_url, params=s_params, timeout=20, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    js = r.json()
    if not js.get("query", {}).get("search"):
        cache[cache_key] = 0.0
        return 0.0

    title = js["query"]["search"][0]["title"]
    # Pageviews (daily)
    end = datetime.utcnow().date() - timedelta(days=1)
    start = end - timedelta(days=days)
    t_enc = requests.utils.quote(title, safe="")
    pv_url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/{t_enc}/daily/{start.strftime('%Y%m%d')}/{end.strftime('%Y%m%d')}"
    pv = requests.get(pv_url, timeout=20, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    if pv.status_code != 200:
        cache[cache_key] = 0.0
        return 0.0
    items = pv.json().get("items", [])
    if not items:
        cache[cache_key] = 0.0
        return 0.0

    avg = sum(i.get("views", 0) for i in items) / max(1, len(items))
    # log-scale normalize
    score = min(1.0, max(0.0, (math.log10(avg + 1) / 6.0) * 1.2))
    cache[cache_key] = score
    return score

def nightlife_penalty(e: Event) -> float:
    # 0..1 penalty
    t = _txt(e)
    p = 0.0
    if "tribute" in t:
        p += 0.10
    if is_nightlife_event(e):
        p += 0.45
    if is_low_signal(e):
        p += 0.35
    return min(0.9, p)

def compelling_score(e: Event, use_wikipedia: bool = True) -> float:
    """Return 0..100"""
    base = venue_prominence(e.venue)  # 0..1
    src = 0.85 if (e.source or "").lower() in ["ticketmaster", "livenation", "pacificsymphony", "scr", "lagunaplayhouse"] else 0.65

    fame = 0.0
    if use_wikipedia:
        q = re.sub(r"\s*\(.*?\)\s*$", "", (e.title or "")).strip()
        fame = wikipedia_pageviews_score(q)

    comp = (0.45 * base) + (0.20 * src) + (0.35 * fame)
    comp = max(0.0, min(1.0, comp - nightlife_penalty(e)))
    return comp * 100.0

def total_score(e: Event, dist_mi: float, comp: float) -> float:
    return 0.55 * comp + 0.45 * proximity_score(dist_mi)

def score_events(events: List[Event], wiki_top_k: int = 60) -> pd.DataFrame:
    """Two-pass scoring:
    1) score without Wikipedia for everyone
    2) compute Wikipedia fame only for the top-k candidates to keep runtime reasonable
    """
    rows = []
    for e in events:
        if e.lat is None or e.lon is None:
            continue
        dist = distance_miles(e.lat, e.lon)
        comp_base = compelling_score(e, use_wikipedia=False)
        total_base = total_score(e, dist_mi=dist, comp=comp_base)
        rows.append({
            "event_obj": e,
            "distance_mi": dist,
            "compelling_base": comp_base,
            "total_base": total_base,
        })
    if not rows:
        return pd.DataFrame()

    rows = sorted(rows, key=lambda r: (r["total_base"], r["compelling_base"]), reverse=True)

    # Wikipedia only for top-k
    enriched = []
    for i, r in enumerate(rows):
        e = r["event_obj"]
        dist = r["distance_mi"]
        if i < wiki_top_k:
            comp = compelling_score(e, use_wikipedia=True)
        else:
            comp = r["compelling_base"]
        total = total_score(e, dist_mi=dist, comp=comp)
        enriched.append({
            **event_to_row(e),
            "distance_mi": round(dist, 1),
            "proximity_score": round(proximity_score(dist), 1),
            "compelling_score": round(comp, 1),
            "total_score": round(total, 1),
            "is_major_venue": bool(is_major_venue_name(e.venue)),
            "major_region": (major_venue_region(e.venue) or ""),
        })

    df = pd.DataFrame(enriched)
    df = df.sort_values(["total_score", "compelling_score"], ascending=False)
    return df


In [ ]:
_BASE_TM = "https://app.ticketmaster.com/discovery/v2/events.json"

# Geohash encoder (minimal, enough for Ticketmaster geoPoint)
_BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
def geohash_encode(lat: float, lon: float, precision: int = 7) -> str:
    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    geohash = []
    bit = 0
    ch = 0
    even = True
    bits = [16, 8, 4, 2, 1]
    while len(geohash) < precision:
        if even:
            mid = sum(lon_interval) / 2
            if lon > mid:
                ch |= bits[bit]
                lon_interval[0] = mid
            else:
                lon_interval[1] = mid
        else:
            mid = sum(lat_interval) / 2
            if lat > mid:
                ch |= bits[bit]
                lat_interval[0] = mid
            else:
                lat_interval[1] = mid
        even = not even
        if bit < 4:
            bit += 1
        else:
            geohash.append(_BASE32[ch])
            bit = 0
            ch = 0
    return "".join(geohash)

def _iso_utc(dt: datetime) -> str:
    return dt.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%dT%H:%M:%SZ")

def fetch_ticketmaster_events(start_dt: datetime, end_dt: datetime, radius_miles: int = 45) -> List[Event]:
    if not TICKETMASTER_API_KEY:
        print("⚠️ Ticketmaster API key not set; skipping Ticketmaster source.")
        return []

    geo = geohash_encode(HOME_LAT, HOME_LON, precision=7)
    params = {
        "apikey": TICKETMASTER_API_KEY,
        "geoPoint": geo,
        "radius": radius_miles,
        "unit": "miles",
        "startDateTime": _iso_utc(start_dt),
        "endDateTime": _iso_utc(end_dt),
        "size": 200,
        "sort": "date,asc",
    }

    events: List[Event] = []
    page = 0
    while True:
        params["page"] = page
        r = requests.get(_BASE_TM, params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"Ticketmaster error {r.status_code}: {r.text[:300]}")
        js = r.json()
        items = js.get("_embedded", {}).get("events", [])
        for it in items:
            name = it.get("name")
            url = it.get("url")
            dt_str = it.get("dates", {}).get("start", {}).get("dateTime") or it.get("dates", {}).get("start", {}).get("localDate")
            start = None
            if dt_str:
                try:
                    # Ticketmaster dateTime is usually UTC; treat as such and convert to local tz
                    start = dateparser.isoparse(dt_str)
                    if start.tzinfo is None:
                        start = start.replace(tzinfo=ZoneInfo("UTC"))
                    start = start.astimezone(ZoneInfo(TZ))
                except Exception:
                    pass
            end = start  # many TM events omit end

            venue_name = None
            city = region = address = None
            lat = lon = None
            try:
                v = it.get("_embedded", {}).get("venues", [])[0]
                venue_name = v.get("name")
                city = v.get("city", {}).get("name")
                region = v.get("state", {}).get("stateCode") or v.get("country", {}).get("countryCode")
                address = v.get("address", {}).get("line1")
                loc = v.get("location") or {}
                if loc.get("latitude") and loc.get("longitude"):
                    lat = float(loc["latitude"])
                    lon = float(loc["longitude"])
            except Exception:
                pass

            # Category mapping via segment name when present
            seg = None
            try:
                seg = it.get("classifications", [])[0].get("segment", {}).get("name")
            except Exception:
                pass
            if seg:
                seg_l = seg.lower()
                if "music" in seg_l:
                    cat = "Concert"
                elif "arts" in seg_l or "theatre" in seg_l or "theater" in seg_l:
                    cat = "Play"
                else:
                    cat = "Event"
            else:
                cat = "Event"

            price_min = price_max = None
            if it.get("priceRanges"):
                pr = it["priceRanges"][0]
                price_min = pr.get("min")
                price_max = pr.get("max")

            e = Event(
                title=name,
                category=cat,
                start=start,
                end=end,
                venue=venue_name,
                city=city,
                region=region,
                address=address,
                lat=lat,
                lon=lon,
                source="Ticketmaster",
                url=url,
                description=None,
                price_min=price_min,
                price_max=price_max,
                tags=seg,
            )
            events.append(e)

        # pagination
        page_info = js.get("page", {})
        if not page_info:
            break
        total_pages = page_info.get("totalPages", 0)
        if page + 1 >= total_pages or page >= 4:  # safety cap
            break
        page += 1
    return events

# --- Major-venue Ticketmaster helpers ---
_BASE_TM_VENUES = "https://app.ticketmaster.com/discovery/v2/venues.json"

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=12),
    retry=retry_if_exception_type((requests.RequestException, RuntimeError)),
)
def _tm_get_json(url: str, params: Dict[str, Any]) -> Dict[str, Any]:
    r = requests.get(url, params=params, timeout=30)
    # Retry on rate-limit and transient server errors
    if r.status_code in (429, 500, 502, 503, 504):
        raise RuntimeError(f"Ticketmaster temporary error {r.status_code}")
    if r.status_code != 200:
        raise RuntimeError(f"Ticketmaster error {r.status_code}: {r.text[:300]}")
    return r.json()

def resolve_ticketmaster_venue_id(keyword: str, city: Optional[str] = None, state_code: str = "CA") -> Optional[str]:
    """Resolve a Ticketmaster venueId from a human venue name (best-effort). Cached."""
    if not TICKETMASTER_API_KEY:
        return None
    cache_key = f"tm:venue_id:{keyword}:{city}:{state_code}"
    if cache_key in cache:
        return cache[cache_key]

    params = {
        "apikey": TICKETMASTER_API_KEY,
        "keyword": keyword,
        "countryCode": "US",
        "stateCode": state_code,
        "size": 20,
    }
    if city:
        params["city"] = city

    js = _tm_get_json(_BASE_TM_VENUES, params)
    venues = js.get("_embedded", {}).get("venues", []) or []
    if not venues:
        cache.set(cache_key, None, expire=7 * 24 * 3600)
        return None

    from difflib import SequenceMatcher
    def _sim(a: str, b: str) -> float:
        return SequenceMatcher(None, (a or "").lower(), (b or "").lower()).ratio()

    best = None
    best_score = -1.0
    for v in venues:
        name = v.get("name", "") or ""
        score = _sim(name, keyword)
        if keyword.lower() in name.lower():
            score += 0.35
        if city:
            vcity = (v.get("city", {}) or {}).get("name", "") or ""
            if vcity and vcity.lower() == city.lower():
                score += 0.25
        # prefer CA venues when present
        st = (v.get("state", {}) or {}).get("stateCode", "") or ""
        if st.upper() == state_code.upper():
            score += 0.05
        if score > best_score:
            best, best_score = v, score

    vid = best.get("id") if best else None
    cache.set(cache_key, vid, expire=30 * 24 * 3600)
    return vid

def fetch_ticketmaster_events_by_venue_id(venue_id: str, start_dt: datetime, end_dt: datetime) -> List[Event]:
    """Fetch Ticketmaster events for a specific venueId in the date window."""
    if not TICKETMASTER_API_KEY:
        return []
    params = {
        "apikey": TICKETMASTER_API_KEY,
        "venueId": venue_id,
        "classificationName": "Music",
        "startDateTime": _iso_utc(start_dt),
        "endDateTime": _iso_utc(end_dt),
        "size": 200,
        "sort": "date,asc",
    }

    events: List[Event] = []
    page = 0
    while True:
        params["page"] = page
        js = _tm_get_json(_BASE_TM, params)
        items = js.get("_embedded", {}).get("events", []) or []

        for it in items:
            name = it.get("name")
            url = it.get("url")

            dt_str = it.get("dates", {}).get("start", {}).get("dateTime") or it.get("dates", {}).get("start", {}).get("localDate")
            start = None
            if dt_str:
                try:
                    start = dateparser.isoparse(dt_str)
                    if start.tzinfo is None:
                        start = start.replace(tzinfo=ZoneInfo("UTC"))
                    start = start.astimezone(ZoneInfo(TZ))
                except Exception:
                    start = None

            venue = city = address = None
            lat = lon = None
            try:
                v0 = it.get("_embedded", {}).get("venues", [])[0]
                venue = v0.get("name")
                city = v0.get("city", {}).get("name")
                address = v0.get("address", {}).get("line1")
                loc = v0.get("location", {})
                lat = float(loc.get("latitude")) if loc.get("latitude") else None
                lon = float(loc.get("longitude")) if loc.get("longitude") else None
            except Exception:
                pass

            price_min = price_max = None
            if it.get("priceRanges"):
                pr = it["priceRanges"][0]
                price_min = pr.get("min")
                price_max = pr.get("max")

            e = Event(
                title=name,
                category="Concert",
                start=start,
                end=None,
                venue=venue,
                city=city,
                address=address,
                lat=lat,
                lon=lon,
                url=url,
                source="Ticketmaster",
                price_min=price_min,
                price_max=price_max,
                tags="Music",
            )
            events.append(e)

        page_info = js.get("page", {}) or {}
        total_pages = page_info.get("totalPages", 0) or 0
        if page + 1 >= total_pages or page >= 4:
            break
        page += 1
    return events

def fetch_ticketmaster_major_venue_concerts(start_dt: datetime, end_dt: datetime) -> List[Event]:
    """Fetch concerts at curated major venues (LA/OC/SD) regardless of radius."""
    if not INCLUDE_MAJOR_VENUE_CONCERTS:
        return []
    if not TICKETMASTER_API_KEY:
        print("⚠️ Ticketmaster API key not set; skipping major venue concerts.")
        return []

    out: List[Event] = []
    for v in MAJOR_CONCERT_VENUES:
        kw = v.get("tm_keyword") or v["name"]
        city = v.get("tm_city")
        try:
            vid = resolve_ticketmaster_venue_id(str(kw), city=city)
        except Exception as ex:
            print(f"TM venueId lookup failed for {v['name']}: {ex}")
            continue
        if not vid:
            continue
        try:
            out += fetch_ticketmaster_events_by_venue_id(vid, start_dt, end_dt)
        except Exception as ex:
            print(f"TM venue events fetch failed for {v['name']}: {ex}")
    return out


In [ ]:
PACIFIC_SYMPHONY_URL = "https://www.pacificsymphony.org/get-tickets"

def scrape_pacific_symphony(start_dt: datetime, end_dt: datetime) -> List[Event]:
    events: List[Event] = []
    r = requests.get(PACIFIC_SYMPHONY_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    for h3 in soup.find_all(["h3", "h2"]):
        title = h3.get_text(" ", strip=True)
        if not title or len(title) < 3:
            continue
        # Heuristic: pacific symphony listing titles are in h3 with nearby li containing date range
        li = h3.find_next("li")
        if not li:
            continue
        dt_text = li.get_text(" ", strip=True)
        if "|" not in dt_text and "AM" not in dt_text and "PM" not in dt_text:
            continue

        # Venue is usually next anchor after li
        venue_a = li.find_next("a")
        venue = venue_a.get_text(" ", strip=True) if venue_a else None

        # Read more link
        read_more = None
        for a in h3.find_all_next("a", limit=10):
            t = a.get_text(" ", strip=True).lower()
            if t in {"read more", "details", "more info"}:
                read_more = a.get("href")
                break

        # Parse date span + time
        parts = [p.strip() for p in dt_text.split("|", 1)]
        date_part = parts[0].strip("• ").strip()
        time_part = parts[1].strip() if len(parts) > 1 else ""
        sd, ed = parse_date_span(date_part)
        tm = None
        if time_part:
            tm = parse_time_text(time_part.split(",")[0].strip())
        start = end = None
        if sd:
            start = datetime.combine(sd, tm or dtime(19, 30), ZoneInfo(TZ))
        if ed:
            end = datetime.combine(ed, tm or dtime(19, 30), ZoneInfo(TZ))

        e = Event(
            title=title,
            category="Concert",
            start=start,
            end=end,
            venue=venue,
            city="Costa Mesa" if venue and "Segerstrom" in venue else None,
            region="CA",
            address=None,
            lat=None,
            lon=None,
            source="PacificSymphony",
            url=read_more if (read_more and read_more.startswith("http")) else (f"https://www.pacificsymphony.org{read_more}" if read_more else None),
            description=None,
        )
        events.append(e)

    # Deduplicate by (title, start_date)
    dedup = {}
    for e in events:
        key = (e.title, e.start.date() if e.start else None)
        dedup[key] = e
    return list(dedup.values())


In [ ]:
LAGUNA_SHOWS_URL = "https://lagunaplayhouse.com/tickets-and-events/shows/"

def scrape_laguna_playhouse(start_dt: datetime, end_dt: datetime) -> List[Event]:
    r = requests.get(LAGUNA_SHOWS_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    events: List[Event] = []
    for h2 in soup.find_all("h2"):
        title = h2.get_text(" ", strip=True)
        if not title or title.lower() in {"shows", "2025 - 2026 productions"}:
            continue

        # Find the first date-looking text after this h2
        date_text = None
        for el in h2.find_all_next(["p", "div", "span"], limit=6):
            txt = el.get_text(" ", strip=True)
            if re.search(r"\b(20\d{2}|19\d{2})\b", txt) or re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", txt, re.I):
                sd, ed = parse_date_span(txt)
                if sd:
                    date_text = txt
                    break

        if not date_text:
            continue

        sd, ed = parse_date_span(date_text)
        start = datetime.combine(sd, dtime(19, 30), ZoneInfo(TZ)) if sd else None
        end = datetime.combine(ed, dtime(22, 0), ZoneInfo(TZ)) if ed else start

        # Buy Tickets link
        buy = None
        for a in h2.find_all_next("a", limit=8):
            if "buy" in a.get_text(" ", strip=True).lower() and "ticket" in a.get_text(" ", strip=True).lower():
                buy = a.get("href")
                break

        e = Event(
            title=title,
            category="Play",
            start=start,
            end=end,
            venue="Laguna Playhouse",
            city="Laguna Beach",
            region="CA",
            address="606 Laguna Canyon Rd, Laguna Beach, CA 92651",
            lat=None,
            lon=None,
            source="LagunaPlayhouse",
            url=buy if (buy and buy.startswith("http")) else (f"https://lagunaplayhouse.com{buy}" if buy else None),
            description=f"Run dates: {date_text}",
        )
        events.append(e)

    # Deduplicate
    dedup = {}
    for e in events:
        dedup[(e.title, e.start.date() if e.start else None)] = e
    return list(dedup.values())


In [ ]:
SCR_SEASON_URL = "https://www.scr.org/plays/plays-landing/2025-26-season-at-a-glance/"

def _absolute(url: str) -> str:
    if not url:
        return url
    if url.startswith("http"):
        return url
    return f"https://www.scr.org{url}"

def extract_scr_production_links() -> List[str]:
    r = requests.get(SCR_SEASON_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/plays/productions/" in href:
            links.add(_absolute(href.split("#")[0]))
    return sorted(links)

def scrape_scr_production(url: str) -> Optional[Event]:
    r = requests.get(url, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    # Title from og:title
    title = None
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        title = og["content"].split("|")[0].strip()
    if not title:
        # fallback to largest heading
        h = soup.find(["h1", "h2"])
        title = h.get_text(" ", strip=True) if h else url

    text = soup.get_text("\n", strip=True)

    # Stage
    stage = None
    for s in ["Segerstrom Stage", "Julianne Argyros Stage", "Nicholas Studio"]:
        if s in text:
            stage = s
            break

    # Age recommendation
    age_hint = None
    m = re.search(r"(Recommendation|Recommended)\s*:?\s*Age\s*(\d{1,2})\s*\+?", text, flags=re.I)
    if m:
        age_hint = f"{m.group(2)}+"
    else:
        m2 = re.search(r"Recommended\s+for\s+ages\s+(\d{1,2})\s*\+?", text, flags=re.I)
        if m2:
            age_hint = f"{m2.group(1)}+"

    # Dates: prefer "Regular Performances:"
    date_span = None
    m = re.search(r"Regular Performances:\s*([^\n]+)", text, flags=re.I)
    if m:
        date_span = m.group(1).strip()
    else:
        # fallback: look for a line containing a year and a month
        for line in text.split("\n"):
            if re.search(r"\b(20\d{2}|19\d{2})\b", line) and re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", line, re.I):
                # avoid "Tickets on Sale" etc
                if "tickets on sale" in line.lower():
                    continue
                date_span = line.strip()
                break

    sd, ed = parse_date_span(date_span or "")
    if not sd:
        return None
    start = datetime.combine(sd, dtime(19, 30), ZoneInfo(TZ))
    end = datetime.combine(ed or sd, dtime(22, 0), ZoneInfo(TZ))

    desc = None
    # short description: first paragraph-ish
    # (keep simple; full text is huge)
    for line in text.split("\n"):
        if len(line) > 80 and "On " not in line[:4]:
            desc = line[:300]
            break

    e = Event(
        title=title,
        category="Play",
        start=start,
        end=end,
        venue="South Coast Repertory" if not stage else stage,
        city="Costa Mesa",
        region="CA",
        address="655 Town Center Dr, Costa Mesa, CA 92628",
        lat=None,
        lon=None,
        source="SCR",
        url=url,
        description=(desc + (f" (Recommended age: {age_hint})" if age_hint else "")) if desc else None,
        age_hint=age_hint,
    )
    return e

def scrape_scr(start_dt: datetime, end_dt: datetime, max_productions: int = 10) -> List[Event]:
    links = extract_scr_production_links()
    # Scrape a limited number (season pages can be big)
    out: List[Event] = []
    for url in links[:max_productions]:
        try:
            ev = scrape_scr_production(url)
            if ev:
                out.append(ev)
        except Exception as ex:
            print(f"SCR scrape failed for {url}: {ex}")
            continue
    return out


In [ ]:
LIVENATION_HOB_URL = "https://www.livenation.com/venue/KovZpZAEA67A/house-of-blues-anaheim-events"

def _parse_livenation_datetime(line: str, tz: str = TZ) -> Optional[datetime]:
    # Example: "Fri Dec 19, 2025 ▪︎ 6:30PM"
    s = line.replace("▪︎", " ").replace("•", " ")
    s = re.sub(r"\s+", " ", s).strip()
    try:
        dt = dateparser.parse(s, fuzzy=True)
        if dt is None:
            return None
        # LiveNation listings are local
        return dt.replace(tzinfo=ZoneInfo(tz)) if dt.tzinfo is None else dt.astimezone(ZoneInfo(tz))
    except Exception:
        return None

def scrape_livenation_house_of_blues(start_dt: datetime, end_dt: datetime) -> List[Event]:
    r = requests.get(LIVENATION_HOB_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    text = soup.get_text("\n", strip=True)

    # Start scanning after "Advertisement" if present
    if "Advertisement" in text:
        text = text.split("Advertisement", 1)[1]

    chunks = text.split("Buy Tickets")
    events: List[Event] = []
    for chunk in chunks[:-1]:
        lines = [ln.strip() for ln in chunk.split("\n") if ln.strip()]
        # remove obvious junk
        lines = [ln for ln in lines if not ln.lower().startswith("image:") and ln.lower() not in {"more info", "today", "tomorrow"}]
        if len(lines) < 4:
            continue

        # Find a line with a weekday + time (best signal)
        dt_line = None
        dt_val = None
        for ln in lines:
            if "am" in ln.lower() or "pm" in ln.lower():
                if re.search(r"\b(mon|tue|wed|thu|fri|sat|sun)\b", ln.lower()):
                    dt_line = ln
                    dt_val = _parse_livenation_datetime(ln)
                    break
        if not dt_val:
            continue

        # Title: pick the most title-like line near dt_line (often right before venue line)
        # We'll take the nearest non-date line above dt_line that isn't the venue
        idx = lines.index(dt_line)
        venue = None
        # Look around for venue-like line
        for ln in lines[max(0, idx-4): idx+3]:
            if "house of blues" in ln.lower():
                venue = ln
                break
        # Candidate title lines: around idx-6..idx
        candidates = []
        for ln in lines[max(0, idx-6): idx]:
            if any(w in ln.lower() for w in ["house of blues", "the parish", "anaheim"]):
                continue
            if re.search(r"\b\d{4}\b", ln) and re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", ln.lower()):
                continue
            if len(ln) < 3 or len(ln) > 120:
                continue
            candidates.append(ln)
        title = candidates[-1] if candidates else None
        if not title:
            continue

        # Event URL: try to find a ticketmaster link nearby
        url = None
        for a in soup.find_all("a", href=True):
            if "ticketmaster.com" in a["href"] and title.lower() in a.get_text(" ", strip=True).lower():
                url = a["href"]
                break

        e = Event(
            title=title,
            category="Concert",
            start=dt_val,
            end=dt_val,
            venue=venue or "House of Blues Anaheim",
            city="Anaheim",
            region="CA",
            address="400 W Disney Way #337, Anaheim, CA 92802",
            source="LiveNation",
            url=url,
        )
        events.append(e)

    # Deduplicate
    dedup = {}
    for e in events:
        key = (e.title, e.start)
        dedup[key] = e
    return list(dedup.values())


In [ ]:
def hydrate_coords(events: List[Event]) -> List[Event]:
    for e in events:
        if e.lat is not None and e.lon is not None:
            continue
        lat, lon = infer_coords(e.venue, e.address)
        if lat is None and e.address:
            try:
                lat, lon = geocode_address(e.address)
            except Exception:
                lat, lon = (None, None)
        e.lat, e.lon = lat, lon
    return events

def filter_events(events: List[Event], window_start: datetime, window_end: datetime) -> List[Event]:
    out = []
    for e in events:
        if not e.title or not e.start:
            continue
        if not overlaps_window(e.start, e.end, window_start, window_end):
            continue
        if is_kids_event(e):
            continue
        if is_promo_event(e):
            continue
        # Nightlife / low-signal filter: aggressively exclude
        if is_nightlife_event(e) and venue_prominence(e.venue) < 0.85:
            continue
        if is_low_signal(e):
            continue
        out.append(e)
    return out

def dedupe(events: List[Event]) -> List[Event]:
    priority = {"Ticketmaster": 3, "PacificSymphony": 2, "SCR": 2, "LagunaPlayhouse": 2, "LiveNation": 1}
    best = {}
    for e in events:
        k = (re.sub(r"\s+", " ", e.title.strip().lower()), e.start.date() if e.start else None)
        p = priority.get(e.source or "", 0)
        if k not in best or p > priority.get(best[k].source or "", 0):
            best[k] = e
    return list(best.values())

# --- Collect ---
events_all: List[Event] = []

try:
    events_all += fetch_ticketmaster_events(START_DT, END_DT, radius_miles=RADIUS_MILES)
except Exception as ex:
    print(f"Ticketmaster (radius) fetch failed: {ex}")

try:
    # Major venues (LA/OC/SD) regardless of radius
    events_all += fetch_ticketmaster_major_venue_concerts(START_DT, END_DT)
except Exception as ex:
    print(f"Ticketmaster (major venues) fetch failed: {ex}")

try:
    events_all += scrape_pacific_symphony(START_DT, END_DT)
except Exception as ex:
    print(f"Pacific Symphony scrape failed: {ex}")

try:
    events_all += scrape_laguna_playhouse(START_DT, END_DT)
except Exception as ex:
    print(f"Laguna Playhouse scrape failed: {ex}")

try:
    events_all += scrape_scr(START_DT, END_DT, max_productions=12)
except Exception as ex:
    print(f"SCR scrape failed: {ex}")

if not TICKETMASTER_API_KEY:
    try:
        events_all += scrape_livenation_house_of_blues(START_DT, END_DT)
    except Exception as ex:
        print(f"LiveNation scrape failed: {ex}")

print(f"Collected raw events: {len(events_all)}")

events_all = hydrate_coords(events_all)
events_all = filter_events(events_all, START_DT, END_DT)
events_all = dedupe(events_all)

print(f"After filtering+dedupe: {len(events_all)}")

df = score_events(events_all, wiki_top_k=60)
df.head()


In [ ]:
def fmt_price(row) -> str:
    if pd.isna(row.get("price_min")) and pd.isna(row.get("price_max")):
        return ""
    lo = row.get("price_min")
    hi = row.get("price_max")
    if pd.notna(lo) and pd.notna(hi) and lo != hi:
        return f"${lo:.0f}–${hi:.0f}"
    if pd.notna(lo):
        return f"from ${lo:.0f}"
    if pd.notna(hi):
        return f"up to ${hi:.0f}"
    return ""

def render_section(df_section: pd.DataFrame, title: str):
    if df_section.empty:
        print(f"\n### {title}\n(no items in top list)")
        return
    display_cols = [
        "title", "start", "venue", "distance_mi", "total_score", "compelling_score", "source", "url"
    ]
    d = df_section[display_cols].copy()
    d["start"] = d["start"].apply(lambda x: x.strftime('%a %b %d, %I:%M %p') if pd.notna(x) else "")
    display(d)

if df.empty:
    print("No events found after filtering. Try increasing radius, adding Ticketmaster API key, or loosening filters.")
else:
    top = df.head(TOP_N).copy()

    # Sectioned output
    print("## Top picks (sectioned)")
    for cat, title in [("Concert", "Concerts"), ("Play", "Plays / Theatre"), ("Event", "Other Events")]:
        render_section(top[top["category"] == cat], title)

    # Also show the full top list in one table
    print("\n## Top list (all categories)")
    display_cols = ["category", "title", "start", "venue", "distance_mi", "total_score", "compelling_score", "source", "url"]
    d = top[display_cols].copy()
    d["start"] = d["start"].apply(lambda x: x.strftime('%a %b %d, %I:%M %p') if pd.notna(x) else "")
    display(d)


## 4) (Optional) Export a shareable HTML report

In [ ]:
from pathlib import Path
from html import escape

def _fmt_clock(dt: datetime) -> str:
    # 9:00 PM style (no leading zero)
    return dt.strftime("%I:%M %p").lstrip("0")

def _fmt_when(start: Optional[datetime], end: Optional[datetime]) -> str:
    if start is None:
        return "TBA"
    if getattr(start, "tzinfo", None) is None:
        start = start.replace(tzinfo=ZoneInfo(TZ))
    s_local = start.astimezone(ZoneInfo(TZ))

    if end is None:
        return f"{s_local:%a %b %d} • {_fmt_clock(s_local)}"
    if getattr(end, "tzinfo", None) is None:
        end = end.replace(tzinfo=ZoneInfo(TZ))
    e_local = end.astimezone(ZoneInfo(TZ))

    if s_local.date() == e_local.date():
        return f"{s_local:%a %b %d} • {_fmt_clock(s_local)}–{_fmt_clock(e_local)}"
    return f"{s_local:%a %b %d} • {_fmt_clock(s_local)} → {e_local:%a %b %d} • {_fmt_clock(e_local)}"

def _fmt_price_row(r: dict) -> str:
    lo = r.get("price_min")
    hi = r.get("price_max")
    try:
        lo = float(lo) if lo is not None and str(lo) != "nan" else None
    except Exception:
        lo = None
    try:
        hi = float(hi) if hi is not None and str(hi) != "nan" else None
    except Exception:
        hi = None
    if lo is None and hi is None:
        return ""
    if lo is not None and hi is not None and abs(lo - hi) < 0.01:
        return f"${lo:,.0f}"
    if lo is not None and hi is not None:
        return f"${lo:,.0f}–${hi:,.0f}"
    if lo is not None:
        return f"from ${lo:,.0f}"
    return f"up to ${hi:,.0f}"

def make_html_report(df_scored: pd.DataFrame, window_start: datetime, window_end: datetime) -> str:
    ws = window_start.strftime("%a %b %d, %Y")
    we = window_end.strftime("%a %b %d, %Y")
    generated = datetime.now(ZoneInfo(TZ)).strftime("%a %b %d, %Y • %I:%M %p %Z").replace(" 0", " ")

    css = """
    :root{
      --bg:#0b1220;
      --card:#121a2b;
      --muted:#9fb0d0;
      --text:#e7eefc;
      --accent:#7aa2ff;
      --accent2:#5eead4;
      --border:rgba(255,255,255,.08);
      --shadow: 0 10px 30px rgba(0,0,0,.25);
      --radius: 18px;
    }
    *{box-sizing:border-box}
    html,body{margin:0;padding:0}
    body{
      font-family: Inter, ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, "Apple Color Emoji","Segoe UI Emoji";
      background: radial-gradient(1200px 600px at 20% 0%, rgba(122,162,255,.18), transparent 55%),
                  radial-gradient(900px 500px at 90% 10%, rgba(94,234,212,.14), transparent 55%),
                  var(--bg);
      color: var(--text);
      line-height: 1.55;
    }
    a{color:inherit;text-decoration:none}
    a:hover{color:var(--accent)}
    .wrap{max-width: 980px; margin: 0 auto; padding: 28px 18px 60px;}
    .topbar{display:flex; gap:14px; align-items:flex-start; justify-content:space-between; flex-wrap:wrap}
    .title{
      font-size: clamp(28px, 3.2vw, 40px);
      letter-spacing: -0.02em;
      margin: 0;
    }
    .subtitle{margin: 8px 0 0; color: var(--muted); font-size: 15px}
    .meta{margin-top: 12px; display:flex; gap:10px; flex-wrap:wrap}
    .pill{
      display:inline-flex; align-items:center; gap:8px;
      padding: 7px 10px;
      border: 1px solid var(--border);
      border-radius: 999px;
      color: var(--muted);
      font-size: 13px;
      background: rgba(255,255,255,.03);
    }
    .pill b{color: var(--text); font-weight: 600}
    .nav{
      margin-top: 18px;
      display:flex; gap:10px; flex-wrap:wrap;
    }
    .nav a{
      padding: 10px 12px;
      border-radius: 999px;
      background: rgba(255,255,255,.05);
      border: 1px solid var(--border);
      font-weight: 600;
      font-size: 13px;
    }
    .nav a:hover{background: rgba(122,162,255,.14); border-color: rgba(122,162,255,.35)}
    .section{margin-top: 26px}
    .section h2{
      margin: 0 0 12px;
      font-size: 20px;
      letter-spacing: -0.01em;
    }
    .grid{
      display:grid;
      grid-template-columns: repeat(1, minmax(0, 1fr));
      gap: 12px;
    }
    @media (min-width: 860px){
      .grid{grid-template-columns: repeat(2, minmax(0, 1fr));}
    }
    .card{
      background: linear-gradient(180deg, rgba(255,255,255,.06), rgba(255,255,255,.03));
      border: 1px solid var(--border);
      border-radius: var(--radius);
      padding: 14px 14px 12px;
      box-shadow: var(--shadow);
      position: relative;
      overflow: hidden;
    }
    .card:before{
      content:"";
      position:absolute; inset:-2px;
      background: radial-gradient(400px 200px at 10% 0%, rgba(122,162,255,.22), transparent 60%),
                  radial-gradient(400px 200px at 90% 10%, rgba(94,234,212,.18), transparent 60%);
      opacity:.4; pointer-events:none;
    }
    .card > *{position:relative}
    .card-title{
      font-weight: 750;
      font-size: 16px;
      letter-spacing: -0.01em;
      margin: 0 0 6px;
    }
    .row{display:flex; gap:10px; flex-wrap:wrap; align-items:center; margin: 8px 0 0}
    .chip{
      padding: 6px 9px;
      border-radius: 999px;
      font-size: 12px;
      border: 1px solid var(--border);
      background: rgba(255,255,255,.03);
      color: var(--muted);
    }
    .chip strong{color: var(--text); font-weight: 650}
    .desc{margin: 10px 0 0; color: rgba(231,238,252,.86); font-size: 13px}
    .muted{color: var(--muted)}
    .rank{
      position:absolute;
      top: 12px; right: 12px;
      font-weight: 800;
      font-size: 12px;
      letter-spacing: .08em;
      padding: 6px 8px;
      border-radius: 10px;
      border: 1px solid rgba(122,162,255,.35);
      background: rgba(122,162,255,.14);
      color: #dbe7ff;
    }
    footer{
      margin-top: 34px;
      color: var(--muted);
      font-size: 13px;
      border-top: 1px solid var(--border);
      padding-top: 16px;
    }
    .small{font-size:12px}
    """

    def build_cards(d: pd.DataFrame, start_rank: int) -> tuple[str, int]:
        cards = []
        rank = start_rank
        for _, r in d.iterrows():
            title = escape(str(r.get("title", "(untitled)")))
            url = str(r.get("url") or "").strip()
            link = f'<a href="{escape(url)}" target="_blank" rel="noopener noreferrer">{title}</a>' if url else title

            when = _fmt_when(r.get("start"), r.get("end"))
            venue = escape(str(r.get("venue") or ""))
            city = escape(str(r.get("city") or ""))
            where = ", ".join([x for x in [venue, city] if x]).strip() or "—"

            dist = r.get("distance_mi")
            dist_s = f"{dist:.1f} mi" if isinstance(dist, (int, float)) else (escape(str(dist)) if dist else "—")

            total = r.get("total_score")
            total_s = f"{total:.1f}" if isinstance(total, (int, float)) else (escape(str(total)) if total else "—")

            comp = r.get("compelling_score")
            comp_s = f"{comp:.1f}" if isinstance(comp, (int, float)) else (escape(str(comp)) if comp else "—")

            price = _fmt_price_row(r)
            src = escape(str(r.get("source") or ""))

            desc = str(r.get("description") or "").strip()
            desc = re.sub(r"\s+", " ", desc)
            if len(desc) > 220:
                desc = desc[:217].rstrip() + "…"
            desc_html = f'<div class="desc">{escape(desc)}</div>' if desc else ""

            chips = [
                f'<span class="chip"><strong>When</strong> {escape(when)}</span>',
                f'<span class="chip"><strong>Where</strong> {where}</span>',
                f'<span class="chip"><strong>Distance</strong> {dist_s}</span>',
                f'<span class="chip"><strong>Score</strong> {total_s} <span class="muted">(comp {comp_s})</span></span>',
            ]
            if price:
                chips.append(f'<span class="chip"><strong>Price</strong> {escape(price)}</span>')
            if src:
                chips.append(f'<span class="chip"><strong>Source</strong> {src}</span>')

            cards.append(f"""
              <article class="card">
                <div class="rank">#{rank}</div>
                <div class="card-title">{link}</div>
                <div class="row">{"".join(chips)}</div>
                {desc_html}
              </article>
            """)
            rank += 1
        return "\n".join(cards), rank

# Sort + slice
df_scored = df_scored.copy() if df_scored is not None else pd.DataFrame()
if not df_scored.empty:
    df_scored = df_scored.sort_values(["total_score", "compelling_score"], ascending=False)

df_top = df_scored.head(TOP_N).copy() if not df_scored.empty else pd.DataFrame()

df_major = pd.DataFrame()
if INCLUDE_MAJOR_VENUE_CONCERTS and (not df_scored.empty) and ("is_major_venue" in df_scored.columns):
    df_major = df_scored[(df_scored["category"] == "Concert") & (df_scored["is_major_venue"] == True)].head(MAJOR_VENUE_TOP_N).copy()

tm_status = "Configured" if (str(TICKETMASTER_API_KEY).strip() != "") else "Not set (limited sources)"

sections = []
rank_cursor = 1

# Optional: spotlight major-venue concerts (LA/OC/SD)
if not df_major.empty:
    sections.append('<section class="section" id="major"><h2>Major venue concerts (LA / OC / San Diego)</h2>')
    cards_html, rank_cursor = build_cards(df_major, rank_cursor)
    sections.append(f'<div class="grid">{cards_html}</div></section>')

for cat, header, anchor in [("Concert", "Concerts", "concerts"),
                            ("Play", "Plays / Theatre", "plays"),
                            ("Event", "Other Events", "events")]:
        d = df_top[df_top["category"] == cat].copy()
        sections.append(f'<section class="section" id="{anchor}"><h2>{header}</h2>')
        if d.empty:
            sections.append('<p class="muted small">(none found in this window)</p></section>')
            continue
        cards_html, rank_cursor = build_cards(d, rank_cursor)
        sections.append(f'<div class="grid">{cards_html}</div></section>')

    nav_links = []
if "df_major" in locals() and not df_major.empty:
    nav_links.append('<a href="#major">Major venues</a>')
nav_links += [
    '<a href="#concerts">Concerts</a>',
    '<a href="#plays">Plays / Theatre</a>',
    '<a href="#events">Other Events</a>',
]
nav_html = "\n".join(nav_links)

html = f"""<!doctype html>
    <html lang="en">
    <head>
      <meta charset="utf-8" />
      <meta name="viewport" content="width=device-width, initial-scale=1" />
      <title>Southern OC — Weekly Event Picks</title>
      <link rel="preconnect" href="https://fonts.googleapis.com">
      <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
      <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;750;800&display=swap" rel="stylesheet">
      <style>{css}</style>
    </head>
    <body>
      <main class="wrap">
        <div class="topbar">
          <div>
            <h1 class="title">Southern OC — Weekly Event Picks</h1>
            <p class="subtitle">Top {len(df_top)} picks for adults &amp; teens • filtered to avoid kid-only, lead-gen “ads”, and low-signal nightlife/restaurant promos.</p>
            <div class="meta">
              <span class="pill"><b>Window</b> {ws} → {we}</span>
              <span class="pill"><b>Home base</b> {escape(HOME_LABEL)}</span>
              <span class="pill"><b>Ticketmaster</b> {escape(tm_status)}</span>
              <span class="pill"><b>Generated</b> {generated}</span>
            </div>
            <nav class="nav" aria-label="Sections">
              {nav_html}
            </nav>
          </div>
        </div>

        {"".join(sections)}

        <footer>
          <div>Scoring combines proximity to {escape(HOME_LABEL)} and a “compelling” signal (venue prominence + optional Wikipedia popularity).</div>
          <div class="small">Tip: if you see something that looks like a class signup/open house, add keywords to the promo filter list in the notebook.</div>
        </footer>
      </main>
    </body>
    </html>
    """
    return html

site_dir = Path("site")
site_dir.mkdir(parents=True, exist_ok=True)
(site_dir / ".nojekyll").write_text("", encoding="utf-8")

out_path = site_dir / "index.html"
out_path.write_text(make_html_report(df, START_DT, END_DT), encoding="utf-8")
print(f"Wrote: {out_path.resolve()}")
out_path


## 5) Scheduling (Tuesday runs)

**Option A (easiest, if you have Colab Pro/Pro+ Scheduled Notebooks):**
- Schedule the notebook to run **daily**, and add a guard cell that only proceeds on **Tuesdays**.  
  (Pro/Pro+ daily scheduling is the common limitation; running daily but exiting early is the workaround.)

**Option B (Colab Enterprise / Google Cloud):**
- Use **Colab Enterprise scheduled notebook runs** in the Google Cloud console to run on a **weekly** schedule.

**Option C (outside Colab):**
- Run the notebook via **papermill** in GitHub Actions / Cloud Run / cron and store the HTML report.
